# Modular Arithmetic Challenge — GPU training (Colab)

Runs the EBM/predictor training stack on a free GPU (~10-15x an M-series Mac).

**Before running:** set the GPU runtime — menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**.

Then **Runtime → Run all**, or run cells top to bottom. Cell 1 confirms the GPU, cell 2 pulls the code, then pick a run (tier 2 finish, or the E2 tier-3 linchpin).

In [ ]:
# 1) Confirm we have a GPU (should print a Tesla T4 table; if it errors, set the GPU runtime above)
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

In [ ]:
# 2) Pull the training code (public fork, ebm-dev branch). Re-run to get the latest.
import os
if not os.path.isdir('modular-arithmetic-challenge'):
    !git clone -q -b ebm-dev https://github.com/eyang07/modular-arithmetic-challenge.git
else:
    !cd modular-arithmetic-challenge && git pull -q
%cd modular-arithmetic-challenge
print('files:', os.listdir('training'))

## (optional) Persist checkpoints to Google Drive
Colab sessions are temporary — when they time out, the cloned repo (and checkpoints) are lost. Mount Drive so checkpoints survive. Skip if you only care about the printed accuracy numbers.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/modchallenge_ckpts', exist_ok=True)
# train.py writes to training/checkpoints/; symlink that to Drive so saves persist.
!rm -rf training/checkpoints && ln -s /content/drive/MyDrive/modchallenge_ckpts training/checkpoints
print('checkpoints will be saved to Google Drive')

## Run A — clear tier 2: per-prime embedding + E6 algebraic-consistency
The decisive tier-2 run. Two new ingredients vs the runs that capped at ~0.71:
- **`cls_pp`**: a learned per-prime embedding (dedicated capacity per prime).
- **E6 (`--alg-consistency`)**: self-supervised ring-axiom losses (commutativity + distributivity). These have **zero loss only when the model computes the true product**, so they push it to *learn the algorithm* instead of memorizing.

**Watch `within-prime-unseen`** — it must reach **≥ 0.90** to clear tier 2. With E6 working, the hope is it climbs alongside `train-fit` (structure generalizes) rather than lagging. ~2–2.5 h on a T4 (E6 adds ~3 forwards/step).

In [ ]:
# Tier 2: per-prime embedding (cls_pp) + E6 algebraic-consistency loss.
# Watch within-prime-unseen -> must reach >=0.90 to clear tier 2.
!python training/train.py --arch cls_pp --tiers 1 2 \
    --fixed-per-prime 2000 --holdout --wd 0.1 --alg-consistency 0.5 \
    --steps 20000 --eval-every 2000 --batch 1024 --lr 5e-4 \
    --d-model 256 --layers 4 --tag t2_clspp_e6

## Run B — E2 linchpin: can we *grok* tier-3 multiplication?
The make-or-break tier-3 test, kept cheap: 8 tier-3 primes, fixed train set, angular head, weight-decay sweep. **Watch `within-prime-unseen`** — if it jumps from ~0 to high (a grokking phase transition) for some weight decay, tier 3 is in reach and we scale up. If it stays flat across all wd after a long run, tier 3 is likely beyond this approach.

`train-fit` should hit ~1.0 early (overfitting the fixed set); the interesting signal is the gap closing.

In [ ]:
# Tier-3 grokking probe: 8 primes, fixed set, angular head, weight-decay sweep.
# ~40 min per wd value on a T4. Watch `within-prime-unseen` for a sudden jump.
for wd in [1.0, 3.0]:
    print(f'\n========== E2: angular, 8 tier-3 primes, wd={wd} ==========')
    !python training/train.py --arch angular --tiers 3 --max-primes 8 \
        --fixed-per-prime 2000 --holdout --wd {wd} \
        --steps 12000 --eval-every 2000 --batch 1024 \
        --d-model 256 --layers 4 --tag e2_ang_wd{wd}

## Download a checkpoint (if you didn't mount Drive)
Lists saved checkpoints and downloads one to your computer.

In [ ]:
import os
print(os.listdir('training/checkpoints'))
# from google.colab import files
# files.download('training/checkpoints/t2_cls_gpu.pt')